#  Graph Processing: Relationships, Traversal, and Iterative Algorithms

**Module**: 12_advanced_topics  
**Notebook**: 07_graph_processing.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

##  Learning Objectives

- **What will I learn?**: How to represent a graph as vertex and edge DataFrames and run degree, connected-component, breadth-first search, and PageRank algorithms with iterative joins.
- **Why does this topic exist?**: Relationships are often the most valuable part of a dataset, and Spark can express many graph algorithms without a separate graph database.
- **What problem does it solve?**: Choosing a graph representation and bounding the shuffle, skew, and lineage cost of iterative processing.
- **Where is it used in real-world Data Engineering?**: Fraud-ring detection, social recommendations, identity resolution, dependency analysis, and logistics networks.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | DataFrames, joins, aggregations, Spark UI, iterative jobs |
| **Required depth** | Deep working knowledge: can model a graph, implement an iterative algorithm, and diagnose its shuffle and lineage cost |
| **Data Engineer** | Build graph features and entity-resolution pipelines with DataFrame joins. |
| **Senior Data Engineer** | Choose between DataFrame iteration, GraphFrames, GraphX, and a dedicated graph system. |
| **Spark specialist** | Reason about Pregel-style iteration, checkpointing, shuffle skew, and supernode handling. |
| **Next topic** | `13_end_to_end_projects/01_batch_etl_pipeline.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Relationships are often the most valuable part of a dataset: who transacts with whom, which accounts
share a device, which services depend on each other. Spark does not ship a Python graph library in the core
distribution. GraphX is a Scala/RDD API, and GraphFrames is a separate package. You can still express many
graph algorithms with vertices and edges as DataFrames and repeated joins, but the iteration changes the cost
profile: every round shuffles state by vertex id, long lineages grow, and a single high-degree vertex can
dominate a partition.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Vertex** | An entity | A row in the vertex DataFrame with a unique id and attributes |
| **Edge** | A relationship | A row with `src`, `dst`, and optional weight or attributes |
| **Degree** | Number of connections | In-degree counts incoming edges; out-degree counts outgoing edges |
| **Connected component** | A group of reachable vertices | A label propagated across undirected edges until stable |
| **BFS** | Shortest path in an unweighted graph | Frontier expansion one hop per iteration |
| **PageRank** | A recursive importance score | Rank flows along outgoing edges with damping and dangling-mass redistribution |
| **Pregel** | Vertex-centric iteration | Each vertex receives messages, updates state, and sends messages in supersteps |
| **Supernode** | A very high-degree vertex | A key that skews one shuffle partition and one task |

### How it works, step by step

1. Represent entities as a vertex DataFrame keyed by a unique id.
2. Represent relationships as an edge DataFrame with `src` and `dst` keys.
3. Each iteration joins the current state to the edges, shuffles by vertex id, and aggregates messages.
4. Connected components propagates the minimum label; BFS propagates the minimum distance; PageRank propagates rank mass.
5. `localCheckpoint` or `checkpoint` truncates the growing logical plan between iterations.
6. Convergence is a business decision: stop when labels stop changing, the frontier is empty, or rank deltas fall below a tolerance.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

###  1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Graph_Processing_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 2. Graph Representation: Vertices, Edges, and Degrees

A graph is two tables:

- A **vertex table** with one row per entity and a unique `id`.
- An **edge table** with `src` and `dst` keys that reference the vertex ids.

The small directed graph below has two components. Vertices 1-6 form one component, and
vertices 7-9 form another. The 4-5-6 cycle is intentional: it makes PageRank interesting
and gives the traversal algorithms a loop to handle.

In [ ]:
vertices = spark.createDataFrame(
    [
        (1, "Alice", "Analytics"),
        (2, "Bob", "Engineering"),
        (3, "Carol", "Engineering"),
        (4, "Dave", "Sales"),
        (5, "Eve", "Sales"),
        (6, "Frank", "Support"),
        (7, "Grace", "Analytics"),
        (8, "Heidi", "Support"),
        (9, "Ivan", "Engineering"),
    ],
    ["id", "name", "team"],
)

edges = spark.createDataFrame(
    [
        (1, 2, "follows"),
        (1, 3, "follows"),
        (2, 3, "follows"),
        (3, 1, "follows"),
        (3, 4, "follows"),
        (4, 5, "follows"),
        (5, 6, "follows"),
        (6, 4, "follows"),
        (7, 8, "follows"),
        (8, 9, "follows"),
    ],
    ["src", "dst", "relationship"],
)

print(f"Vertices: {vertices.count()} | Edges: {edges.count()}")
vertices.show()
edges.show()

## 3. Degrees: The Simplest Graph Metric

In-degree counts incoming edges; out-degree counts outgoing edges. Degree is cheap to
compute and often reveals hubs, sinks, and likely supernodes before a more expensive
algorithm runs.

In [ ]:
out_degrees = edges.groupBy(F.col("src").alias("id")).agg(F.count("*").alias("out_degree"))
in_degrees = edges.groupBy(F.col("dst").alias("id")).agg(F.count("*").alias("in_degree"))

degrees = (
    vertices.join(out_degrees, "id", "left")
    .join(in_degrees, "id", "left")
    .fillna(0)
    .select("id", "name", "team", "in_degree", "out_degree")
)

degrees.orderBy(F.desc("in_degree"), "id").show()

## 4. Connected Components by Label Propagation

Connected components group vertices that are reachable from one another. The algorithm
starts with every vertex in its own component and repeatedly propagates the smallest
label across **undirected** edges until no label changes.

Each round is a join, a shuffle by vertex id, and an aggregation. `localCheckpoint` cuts
the growing logical plan between rounds. Without it, a long loop can produce a very deep
plan and exhaust the driver or the JVM. `localCheckpoint` is fast but not durable: if an
executor is lost, its checkpoint blocks can be lost too. Use `checkpoint()` to reliable
storage when the iteration must survive executor failure.

In [ ]:
undirected = (
    edges.select("src", "dst")
    .union(edges.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
    .distinct()
)

labels = vertices.select("id", F.col("id").alias("component"))

for iteration in range(1, 11):
    proposals = (
        labels.alias("l")
        .join(undirected.alias("e"), F.col("l.id") == F.col("e.src"))
        .select(F.col("e.dst").alias("id"), F.col("l.component").alias("proposed"))
    )
    updated = labels.union(proposals).groupBy("id").agg(F.min("component").alias("component"))
    changed = (
        labels.alias("old")
        .join(updated.alias("new"), F.col("old.id") == F.col("new.id"))
        .filter(F.col("old.component") != F.col("new.component"))
        .count()
    )
    labels = updated.localCheckpoint(eager=True)
    print(f"Iteration {iteration}: {changed} label changes")
    if changed == 0:
        break

labels.orderBy("component", "id").show()

## 5. Breadth-First Search for Unweighted Shortest Paths

BFS expands a frontier one hop at a time. The first time a vertex is reached is its
shortest distance in an unweighted graph. The `left_anti` join prevents already-visited
vertices from re-entering the frontier, which also prevents infinite loops on cycles.

In [ ]:
source_id = 1
frontier = spark.createDataFrame([(source_id, 0)], ["id", "distance"])
visited = frontier

for iteration in range(1, 10):
    next_frontier = (
        frontier.alias("f")
        .join(edges.alias("e"), F.col("f.id") == F.col("e.src"))
        .select(F.col("e.dst").alias("id"), (F.col("f.distance") + 1).alias("distance"))
        .distinct()
        .join(visited, "id", "left_anti")
    )
    new_vertices = next_frontier.count()
    print(f"Iteration {iteration}: {new_vertices} newly reached vertices")
    if new_vertices == 0:
        break
    visited = visited.union(next_frontier)
    frontier = next_frontier

visited.orderBy("distance", "id").show()

## 6. PageRank with Dangling Nodes

PageRank models a random walk: with probability `d` the walk follows an outgoing edge,
and with probability `1 - d` it jumps to a random vertex. A vertex with no outgoing edges
is a **dangling node**. Its rank has nowhere to flow, so it is redistributed uniformly
across all vertices. Omitting that step makes total rank drift below 1.0 and biases every
score.

The implementation below uses `localCheckpoint(eager=True)` on every iteration. This is
not cosmetic: without checkpointing, 20 iterations build a deep plan and can exhaust the
driver heap even on a nine-vertex graph. Remember that a local checkpoint is not durable;
use a reliable `checkpoint()` when the job must recover after losing an executor.

In [ ]:
def pagerank(edge_df, vertex_df, iterations=20, damping=0.85):
    n = vertex_df.count()
    ranks = vertex_df.select("id", F.lit(1.0 / n).alias("rank"))
    out_degrees = edge_df.groupBy(F.col("src").alias("id")).agg(F.count("*").alias("out_degree"))

    for _ in range(iterations):
        contributions = (
            edge_df.alias("e")
            .join(ranks.alias("r"), F.col("e.src") == F.col("r.id"))
            .join(out_degrees.alias("o"), F.col("e.src") == F.col("o.id"))
            .select(
                F.col("e.dst").alias("id"),
                (F.col("r.rank") / F.col("o.out_degree")).alias("contribution"),
            )
        )
        incoming = contributions.groupBy("id").agg(F.sum("contribution").alias("incoming"))
        dangling = (
            ranks.alias("r")
            .join(out_degrees.alias("o"), F.col("r.id") == F.col("o.id"), "left")
            .filter(F.col("o.out_degree").isNull())
            .agg(F.coalesce(F.sum("r.rank"), F.lit(0.0)))
            .first()[0]
        )
        ranks = (
            ranks.select("id")
            .join(incoming, "id", "left")
            .fillna(0.0, subset=["incoming"])
            .select(
                "id",
                ((1 - damping) / n + damping * (F.col("incoming") + dangling / n)).alias("rank"),
            )
            .localCheckpoint(eager=True)
        )
    return ranks


ranks = pagerank(edges, vertices, iterations=20)
ranked = (
    ranks.join(vertices, "id")
    .select("id", "name", "team", F.round("rank", 4).alias("rank"))
    .orderBy(F.desc("rank"))
)
ranked.show()

---

##  Experiment: Convergence of Iterative PageRank

- **1. What are we testing?**: Compare PageRank after one iteration and after 20 iterations on the same graph.
- **2. What do we expect?**: One iteration ranks nodes mostly by direct incoming links; later iterations move rank through longer paths and the 4-5-6 cycle.

In [ ]:
rank_1 = pagerank(edges, vertices, iterations=1)
rank_20 = pagerank(edges, vertices, iterations=20)

comparison = (
    rank_1.alias("one")
    .join(rank_20.alias("twenty"), "id")
    .select(
        "id",
        F.round(F.col("one.rank"), 4).alias("rank_1"),
        F.round(F.col("twenty.rank"), 4).alias("rank_20"),
        F.round(F.abs(F.col("one.rank") - F.col("twenty.rank")), 4).alias("abs_delta"),
    )
    .orderBy(F.desc("rank_20"))
)
comparison.show()
print("Sum of ranks after 20 iterations:", rank_20.agg(F.sum("rank")).first()[0])

**3. What happened?**: The largest per-node change between one and 20 iterations is about 0.107. Vertices 4, 5, and 6 rise as rank circulates through the 4-5-6 cycle, and the 20-iteration ranks sum to approximately 1.0.  
**4. Why did it happen?**: PageRank is a fixed-point iteration. Each round propagates rank one hop, and the damping factor reduces the mass that has not yet reached a stable distribution.  
**5. Practical Engineering Lesson**: Choose an iteration count or convergence tolerance deliberately, checkpoint each round, and do not present a one-iteration ranking as converged.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Fraud-ring detection

- **Industry**: Financial services
- **Scenario**: Analysts need to find groups of accounts that share devices, cards, or addresses and may be part of the same fraud ring.
- **Problem**: Looking at one account at a time hides the shared identifiers that connect a ring.
- **Why this concept is relevant**: Connected components over shared-identifier edges collapse each ring into one reviewable group.
- **Implementation idea**: Build vertices for accounts and identifiers, add edges for each observed link, propagate component labels, and rank components by size and risk.
- **Expected result**: A short list of connected groups for manual review, with the shared identifiers preserved as evidence.
- **Alternative**: A graph database with native traversal.
- **Trade-offs**: A DataFrame implementation keeps the pipeline in Spark but requires checkpointing and careful handling of high-degree identifiers.

### Use Case 2: Social recommendation

- **Industry**: Media and social platforms
- **Scenario**: A product team wants to rank accounts by influence in a follow graph and recommend who to follow next.
- **Problem**: Raw follower counts reward already-famous accounts and ignore where attention flows through the network.
- **Why this concept is relevant**: PageRank or degree-based features capture both direct and indirect importance, and can be joined to candidate recommendations.
- **Implementation idea**: Build a directed follow graph, run PageRank for a bounded number of iterations, and join the scores to candidate pairs.
- **Expected result**: A ranked candidate list with an explainable network feature.
- **Alternative**: A dedicated graph engine or a feature store with precomputed graph embeddings.
- **Trade-offs**: Iterative DataFrame PageRank is transparent and dependency-free, but repeated full-graph runs are expensive at very large scale.

### Use Case 3: Identity resolution

- **Industry**: Marketing and customer data
- **Scenario**: Customer records arrive from multiple systems with different ids, emails, phones, and device identifiers.
- **Problem**: Duplicate profiles inflate counts and split the customer journey across records.
- **Why this concept is relevant**: Connected components over matching identifiers produce a canonical entity id for each cluster.
- **Implementation idea**: Normalize identifiers, create edges for exact or probabilistic matches, propagate component labels, and persist the mapping.
- **Expected result**: A canonical id mapping that downstream marts can join to.
- **Alternative**: A master data management platform or a specialized entity-resolution tool.
- **Trade-offs**: A Spark pipeline is flexible and auditable, but probabilistic matching needs threshold tuning and human review.

---

## Business Value

- Finds fraud rings and shared-identity clusters that row-level rules miss.
- Improves recommendation and influence features with network structure.
- Reduces duplicate customer records and inconsistent reporting.
- Makes the cost of iterative graph processing visible before committing to a platform.
- Provides a dependency-free baseline before adopting GraphFrames, GraphX, or a graph database.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Bound iterations and define a convergence tolerance; do not loop forever.
- Checkpoint long iterative plans to truncate lineage.
- Use `localCheckpoint` for fast lineage truncation and reliable `checkpoint()` when recovery across executor loss matters.
- Partition or salt high-degree vertices to reduce shuffle skew.
- Redistribute dangling-node mass in PageRank and verify that scores sum to approximately 1.0.
- Measure shuffle read/write, task duration skew, and checkpoint I/O for every extra round.
- Use GraphFrames, GraphX, or a dedicated graph system when the algorithm or scale justifies the dependency.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Convergence of iterative PageRank

**Steps**

1. Run PageRank for one iteration and for 20 iterations.
2. Compare the per-node ranks and the largest absolute change.
3. Check that the 20-iteration ranks sum to approximately 1.0.

**What to observe**: One iteration ranks nodes mostly by direct incoming links; later iterations move mass through longer paths and the 4-5-6 cycle.

**Healthy result**: Ranks stabilize and sum to about 1.0; the largest change between 1 and 20 iterations is about 0.107 on this graph.

**Problematic result**: Ranks keep moving, the sum drifts from 1.0, or the job slows because the plan grows.

**Engineering takeaway**: Pick a convergence tolerance, checkpoint each iteration, and measure the cost of every extra round.

---

## Trade-offs and Alternatives

### Advantages

- Uses the existing DataFrame API and cluster
- No extra runtime dependency
- Easy to inspect plans and intermediate state

### Limitations

- Iterative joins are expensive
- No built-in graph algorithms in PySpark
- Supernodes cause skew
- Convergence and correctness are your responsibility

### When to use it

- Entity resolution
- Fraud rings
- Dependency analysis
- Moderate-size graphs

### When not to use it

- Billion-edge low-latency traversal
- Algorithms that need a specialized graph engine
- Repeated interactive graph queries

### Alternatives inside Spark

- DataFrame iterative joins
- GraphX (Scala/RDD)
- GraphFrames (separate package)

### Alternatives outside Spark

- Neo4j
- Amazon Neptune
- TigerGraph
- NetworkX for single-node graphs

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

##  Practical Exercises

###  Exercise 1 (Beginner): Find the Most Connected Vertices
**Problem Statement**: Use the degree table to list the three vertices with the highest in-degree.  
**Input / Context**: The `degrees` DataFrame

In [ ]:
# Write your solution for Exercise 1 here

###  Exercise 2 (Intermediate): Traverse the Second Component
**Problem Statement**: Change the BFS source to vertex 7 and explain why vertices 1-6 are absent from the result.  
**Input / Context**: The BFS cell and the `edges` DataFrame

In [ ]:
# Write your solution for Exercise 2 here

###  Exercise 3 (Advanced): Add a Hub and Inspect Skew
**Problem Statement**: Add vertex 10 and connect it to every other vertex. Explain why the first join becomes skewed and propose one mitigation.  
**Input / Context**: The `vertices` and `edges` DataFrames

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Connected components never converge

**Scenario**: A connected-components job keeps running, and the Spark UI shows one task processing far more shuffle data than the others.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the graph is directed and whether edges are mirrored before propagation.
2. Look for a high-degree vertex and inspect partition sizes.
3. Verify that each iteration truncates lineage and that the convergence condition is reachable.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

##  Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Find the Most Connected Vertices):
```python
degrees.orderBy(F.desc("in_degree")).show(3)
```

#### Solution 2 (Traverse the Second Component):
```python
source_id = 7
# Re-run the BFS loop. Only vertices 7, 8, and 9 are reachable.
# The graph has two disconnected components, so the frontier never reaches 1-6.
```

#### Solution 3 (Add a Hub and Inspect Skew):
```python
hub_edges = spark.createDataFrame(
    [(10, i, "hub") for i in range(1, 10)],
    ["src", "dst", "relationship"],
)
edges_with_hub = edges.union(hub_edges)
# All edges with src = 10 hash to one partition, so one task receives every hub edge.
# Salt the hub key, split it into virtual nodes, or use a graph engine that handles high-degree vertices.
```

</details>

---

##  Module Summary & Interview Guide

###  Key Takeaways
- **Graphs as tables**: Vertices and edges are DataFrames; joins express traversal and message passing.
- **Iteration is explicit**: Connected components, BFS, and PageRank loop until a convergence condition is met.
- **Checkpoint long lineages**: Each round adds to the logical plan; localCheckpoint or checkpoint truncates it.
- **Know the ecosystem**: GraphX is a Scala/RDD API; GraphFrames is a separate DataFrame package; verify compatibility before adding either.

###  Essential Technical Terms
- `Vertex`: An entity with a unique id.
- `Edge`: A directed or undirected relationship between two vertices.
- `Degree`: The number of incoming or outgoing edges.
- `Connected component`: A maximal set of vertices reachable from one another.
- `BFS`: Frontier expansion that finds unweighted shortest paths.
- `PageRank`: An iterative importance score that flows along edges.
- `Pregel`: A vertex-centric model of supersteps with message passing.
- `Supernode`: A high-degree vertex that can skew one shuffle partition.
- `Dangling node`: A vertex with no outgoing edges in a directed graph.

###  Common Engineering Mistakes
-  **Treating a directed graph as undirected**: Mirror the edges only when the algorithm needs undirected connectivity.
-  **Ignoring dangling nodes**: PageRank mass leaks unless it is redistributed.
-  **Iterating without checkpointing**: The logical plan grows every round and can exhaust the driver.
-  **Ignoring supernodes**: One key can dominate a shuffle partition and leave the rest of the cluster idle.

###  Interview Questions on This Topic
-  **Q: How would you represent a graph in Spark?**
  - *A*: As a vertex DataFrame keyed by id and an edge DataFrame with src and dst. Algorithms join the current state to the edges and aggregate messages, iterating until convergence.
-  **Q: What is the Pregel model, and how does it map to Spark?**
  - *A*: Pregel is vertex-centric iteration in supersteps: each vertex receives messages, updates its state, and sends messages to neighbors. In Spark, each superstep is a join and aggregation; GraphX implements it directly, while DataFrame code emulates it.
-  **Q: Why can PageRank fail on a graph with dangling nodes?**
  - *A*: Dangling vertices have no outgoing edges, so their rank is never propagated. Without redistributing that mass, total rank drifts below 1.0 and the scores are biased.

###  What to Study Next
Module 12 is complete. Continue to `13_end_to_end_projects/01_batch_etl_pipeline.ipynb`.

---

###  Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

### More Interview Questions

- **Q: How would you represent a graph in Spark?**
  - *A*: As a vertex DataFrame keyed by id and an edge DataFrame with src and dst. Algorithms join state to edges and aggregate messages, iterating until convergence.
- **Q: What is the Pregel model, and how does it map to Spark?**
  - *A*: Pregel is vertex-centric iteration in supersteps: each vertex receives messages, updates its state, and sends messages to neighbors. In Spark, each superstep is a join and aggregation; GraphX implements it directly, while DataFrame code emulates it.
- **Q: Why can PageRank fail on a graph with dangling nodes?**
  - *A*: Dangling vertices have no outgoing edges, so their rank is never propagated. Without redistributing that mass, total rank drifts below 1.0 and the scores are biased.


In [ ]:
spark.stop()
print('SparkSession successfully stopped.')